# BB + RSI Baseline Strategy

This notebook evaluates the **Bollinger Bands + RSI** baseline strategy
on BTC/USDT and ETH/USDT using historical OHLCV data.

Strategy implementation:
`strategies/baselines/bb_rsi.py`

This notebook focuses on:
- Strategy performance
- Cumulative returns
- Drawdown analysis

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from strategies.baseline.bb_rsi import (
    compute_indicators,
    generate_signals,
    deduplicate_signals,
)

In [ ]:
def evaluate_strategy(df):
    df = df.copy()

    df["returns"] = df["close"].pct_change()
    df["strategy_returns"] = df["signal"].shift(1) * df["returns"]

    df["cumulative_market_returns"] = (1 + df["returns"]).cumprod()
    df["cumulative_strategy_returns"] = (1 + df["strategy_returns"]).cumprod()

    df["drawdown"] = (
        df["cumulative_strategy_returns"]
        / df["cumulative_strategy_returns"].cummax()
        - 1
    )

    return df


def plot_results(df, title):
    fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(12, 14))

    ax1.plot(df.index, df["strategy_returns"], label="Strategy Returns")
    ax1.set_title(f"{title} – Strategy Returns")
    ax1.legend()

    ax2.plot(df.index, df["cumulative_market_returns"], label="Market")
    ax2.plot(df.index, df["cumulative_strategy_returns"], label="Strategy")
    ax2.set_title(f"{title} – Cumulative Returns")
    ax2.legend()

    ax3.plot(df.index, df["drawdown"], color="red")
    ax3.set_title(f"{title} – Drawdown")

    plt.tight_layout()
    plt.show()


In [ ]:
# Load BTC data
btc = pd.read_csv("data/processed/btc_1d.csv")
btc["datetime"] = pd.to_datetime(btc["datetime"])
btc.set_index("datetime", inplace=True)

# Apply BB + RSI strategy
btc = compute_indicators(btc)
btc = generate_signals(btc)
btc = deduplicate_signals(btc)

# Evaluate
btc = evaluate_strategy(btc)

# Plot
plot_results(btc, "BTC/USDT – BB + RSI")

In [ ]:
# Load ETH data
eth = pd.read_csv("data/processed/eth_1d.csv")
eth["datetime"] = pd.to_datetime(eth["datetime"])
eth.set_index("datetime", inplace=True)

# Apply BB + RSI strategy
eth = compute_indicators(eth)
eth = generate_signals(eth)
eth = deduplicate_signals(eth)

# Evaluate
eth = evaluate_strategy(eth)

# Plot
plot_results(eth, "ETH/USDT – BB + RSI")

In [ ]:
def summary_metrics(df):
    return {
        "Total Return": df["cumulative_strategy_returns"].iloc[-1] - 1,
        "Max Drawdown": df["drawdown"].min(),
        "Sharpe (naive)": (
            df["strategy_returns"].mean()
            / df["strategy_returns"].std()
        ) * np.sqrt(252),
    }

print("BTC Metrics:", summary_metrics(btc))
print("ETH Metrics:", summary_metrics(eth))